# IPL CASE STUDY - Data Pipeline Design and Analysis

## Prerequisite

#### Install pandas

In [26]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Usecases

### Task 1: Data Ingestion

In [ ]:
from pathlib import Path
import pandas as pd


deliveries_path = Path("../dataset") / "deliveries.csv"
matches_path = Path("../dataset") / "matches.csv"

if not deliveries_path.exists():
    deliveries_path = Path("deliveries.csv")

if not matches_path.exists():
    matches_path = Path("matches.csv")

deliveries_df = pd.read_csv(deliveries_path)
matches_df = pd.read_csv(matches_path)


def inspect_frame(name, frame):
    print(f"[{name} Shape]: {frame.shape}")
    print(f"[{name} Columns]: {frame.columns.tolist()}")
    print(f"[{name} Dtypes]:")
    print(frame.dtypes)
    print()


inspect_frame("Deliveries", deliveries_df)
inspect_frame("Matches", matches_df)

[Deliveries Shape]: (260920, 17)
[Deliveries Columns]: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']
[Deliveries Dtypes]:
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind        str
fielder               str
dtype: object

[Matches Shape]: (1095, 20)
[Matches Columns]: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 

### Task 2: Data Cleaning & Validation

In [28]:
print("[Missing Values in Deliveries]:")
print(deliveries_df.isna().sum()[deliveries_df.isna().sum() > 0])
print()
print("[Missing Values in Matches]:")
print(matches_df.isna().sum()[matches_df.isna().sum() > 0])
print()

deliveries_df["extras_type"] = deliveries_df["extras_type"].fillna("none")
deliveries_df["player_dismissed"] = deliveries_df["player_dismissed"].fillna("None")
deliveries_df["dismissal_kind"] = deliveries_df["dismissal_kind"].fillna("None")
deliveries_df["fielder"] = deliveries_df["fielder"].fillna("None")
matches_df["city"] = matches_df["city"].fillna("Unknown")
matches_df["player_of_match"] = matches_df["player_of_match"].fillna("Unknown")
matches_df["winner"] = matches_df["winner"].fillna("No Result")

numeric_columns = ["match_id", "inning", "over", "ball", "batsman_runs", "extra_runs", "total_runs", "is_wicket"]
for column in numeric_columns:
    if column in deliveries_df.columns:
        deliveries_df[column] = pd.to_numeric(deliveries_df[column], errors="coerce")

for column in ["id", "result_margin", "target_runs", "target_overs"]:
    if column in matches_df.columns:
        matches_df[column] = pd.to_numeric(matches_df[column], errors="coerce")

deliveries_df = deliveries_df.drop_duplicates()
matches_df = matches_df.drop_duplicates()

categorical_delivery_columns = ["batting_team", "bowling_team", "batter", "bowler", "non_striker", "extras_type", "player_dismissed", "dismissal_kind", "fielder"]
for column in categorical_delivery_columns:
    if column in deliveries_df.columns:
        deliveries_df[column] = deliveries_df[column].astype("category")

categorical_match_columns = ["season", "city", "match_type", "player_of_match", "venue", "team1", "team2", "toss_winner", "toss_decision", "winner", "result", "super_over", "method", "umpire1", "umpire2"]
for column in categorical_match_columns:
    if column in matches_df.columns:
        matches_df[column] = matches_df[column].astype("category")

delivery_match_ids = set(deliveries_df["match_id"].dropna().astype(int))
match_ids = set(matches_df["id"].dropna().astype(int))
unmatched_delivery_ids = delivery_match_ids - match_ids
unmatched_match_ids = match_ids - delivery_match_ids

print(f"[Unmatched Delivery Match IDs]: {len(unmatched_delivery_ids)}")
print(f"[Unmatched Match IDs]: {len(unmatched_match_ids)}")
print("Data cleaning and validation complete.")

[Missing Values in Deliveries]:
extras_type         246795
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

[Missing Values in Matches]:
city                 51
player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64

[Unmatched Delivery Match IDs]: 0
[Unmatched Match IDs]: 0
Data cleaning and validation complete.


### Task 3: Data Transformation

In [29]:
deliveries_df.columns = deliveries_df.columns.str.strip().str.lower().str.replace(" ", "_")
matches_df.columns = matches_df.columns.str.strip().str.lower().str.replace(" ", "_")

deliveries_df["total_runs_per_ball"] = deliveries_df["batsman_runs"].fillna(0) + deliveries_df["extra_runs"].fillna(0)
deliveries_df["total_runs"] = deliveries_df["total_runs"].fillna(deliveries_df["total_runs_per_ball"])
deliveries_df["over_number"] = pd.to_numeric(deliveries_df["over"], errors="coerce").fillna(0).astype(int) + 1

unified_df = deliveries_df.merge(matches_df, left_on="match_id", right_on="id", how="inner")

print(f"[Unified DataFrame Shape]: {unified_df.shape}")

[Unified DataFrame Shape]: (260920, 39)


### Task 4: Core Analysis

In [30]:
runs_per_match = unified_df.groupby("match_id", as_index=False)["total_runs"].sum().rename(columns={"total_runs": "total_match_runs"}).sort_values("total_match_runs", ascending=False)

team_scores = unified_df.groupby(["match_id", "batting_team"], as_index=False)["total_runs"].sum().sort_values(["match_id", "total_runs"], ascending=[True, False])

top_batters = unified_df.groupby("batter", as_index=False)["batsman_runs"].sum().sort_values("batsman_runs", ascending=False).head(10)

batter_stats = unified_df.groupby("batter", as_index=False).agg(total_runs=("batsman_runs", "sum"), balls_faced=("batter", "count"))
batter_stats["strike_rate"] = batter_stats["total_runs"].div(batter_stats["balls_faced"]).mul(100)
strike_rate_df = batter_stats[batter_stats["balls_faced"] >= 200].sort_values("strike_rate", ascending=False).head(10)

bowler_stats = unified_df.groupby("bowler", as_index=False).agg(runs_conceded=("total_runs", "sum"), balls_bowled=("bowler", "count"))
bowler_stats["overs_bowled"] = bowler_stats["balls_bowled"].div(6)
bowler_stats["economy_rate"] = bowler_stats["runs_conceded"].div(bowler_stats["overs_bowled"])
economy_df = bowler_stats[bowler_stats["overs_bowled"] >= 50].sort_values("economy_rate", ascending=True).head(10)

matches_played = unified_df.groupby("batter", as_index=False)["match_id"].nunique().rename(columns={"match_id": "matches_played"})
consistency_df = batter_stats.merge(matches_played, on="batter", how="left")
consistency_df["avg_runs"] = consistency_df["total_runs"].div(consistency_df["matches_played"])
consistent_batters = consistency_df[consistency_df["matches_played"] >= 30].sort_values("avg_runs", ascending=False).head(10)

highest_score = unified_df.groupby(["match_id", "batter"], as_index=False)["batsman_runs"].sum().sort_values("batsman_runs", ascending=False).head(1)

fours_count = int((unified_df["batsman_runs"] == 4).sum())
sixes_count = int((unified_df["batsman_runs"] == 6).sum())
boundaries = unified_df[unified_df["batsman_runs"].isin([4, 6])]
total_boundaries = len(boundaries)
top_boundary_hitters = boundaries.groupby("batter", as_index=False).size().rename(columns={"size": "boundary_count"}).sort_values("boundary_count", ascending=False).head(5)
boundary_runs = boundaries.groupby("batter", as_index=False)["batsman_runs"].sum().rename(columns={"batsman_runs": "runs_from_boundaries"})
boundary_pct_df = batter_stats.merge(boundary_runs, on="batter", how="left").fillna({"runs_from_boundaries": 0})
boundary_pct_df["boundary_percentage"] = boundary_pct_df["runs_from_boundaries"].div(boundary_pct_df["total_runs"].replace(0, pd.NA)).mul(100).fillna(0)

dot_balls = unified_df[unified_df["total_runs"] == 0]
top_dot_bowlers = dot_balls.groupby("bowler", as_index=False).size().rename(columns={"size": "dot_balls"}).sort_values("dot_balls", ascending=False).head(5)

over_totals = unified_df.groupby(["match_id", "over_number"], as_index=False)["total_runs"].sum()
runs_per_over = over_totals.groupby("over_number", as_index=False)["total_runs"].mean().sort_values("over_number")

powerplay = unified_df[unified_df["over_number"].between(1, 6)]
powerplay_teams = powerplay.groupby("batting_team", as_index=False)["total_runs"].sum().sort_values("total_runs", ascending=False)

death_overs = unified_df[unified_df["over_number"].between(16, 20)]
death_overs_teams = death_overs.groupby("batting_team", as_index=False)["total_runs"].sum().sort_values("total_runs", ascending=False)
death_overs_batters = death_overs.groupby("batter", as_index=False)["batsman_runs"].sum().sort_values("batsman_runs", ascending=False).head(10)

inning_runs = unified_df.groupby(["match_id", "inning"], as_index=False)["total_runs"].sum()
inning_distribution = inning_runs.groupby("inning", as_index=False).agg(total_runs=("total_runs", "sum"), avg_match_runs=("total_runs", "mean")).sort_values("inning")

toss_team_scores = team_scores.merge(matches_df[["id", "toss_winner", "winner"]], left_on="match_id", right_on="id", how="left")
toss_winner_scores = toss_team_scores[toss_team_scores["batting_team"].astype(str) == toss_team_scores["toss_winner"].astype(str)]
opponent_scores = toss_team_scores[toss_team_scores["batting_team"].astype(str) != toss_team_scores["toss_winner"].astype(str)]
toss_win_count = int((matches_df["toss_winner"].astype(str) == matches_df["winner"].astype(str)).sum())
toss_win_pct = (toss_win_count / len(matches_df) * 100) if len(matches_df) else 0
toss_runs_comparison = pd.DataFrame({"group": ["toss_winner", "opponent"], "matches": [toss_winner_scores["match_id"].nunique(), opponent_scores["match_id"].nunique()], "avg_runs": [toss_winner_scores["total_runs"].mean(), opponent_scores["total_runs"].mean()]})

match_top_batters = unified_df.groupby(["match_id", "batter"], as_index=False)["batsman_runs"].sum().sort_values(["match_id", "batsman_runs"], ascending=[True, False]).drop_duplicates("match_id")
match_top_wicket_takers = deliveries_df[deliveries_df["is_wicket"] == 1].groupby(["match_id", "bowler"], as_index=False).size().rename(columns={"size": "wickets"}).sort_values(["match_id", "wickets"], ascending=[True, False]).drop_duplicates("match_id")
pom_summary = matches_df[["id", "player_of_match"]].merge(match_top_batters, left_on="id", right_on="match_id", how="left").merge(match_top_wicket_takers, left_on="id", right_on="match_id", how="left", suffixes=("_bat", "_bowl"))
pom_top_batter_count = int((pom_summary["player_of_match"].astype(str) == pom_summary["batter"].astype(str)).sum())
pom_top_bowler_count = int((pom_summary["player_of_match"].astype(str) == pom_summary["bowler"].astype(str)).sum())
pom_top_contributor_count = int(((pom_summary["player_of_match"].astype(str) == pom_summary["batter"].astype(str)) | (pom_summary["player_of_match"].astype(str) == pom_summary["bowler"].astype(str))).sum())

venue_stats = unified_df.groupby(["match_id", "venue"], as_index=False)["total_runs"].sum()
venue_avg = venue_stats.groupby("venue", as_index=False).agg(matches_hosted=("match_id", "count"), avg_runs=("total_runs", "mean")).sort_values("avg_runs", ascending=False)

city_stats = unified_df.groupby(["match_id", "city"], as_index=False)["total_runs"].sum()
city_avg = city_stats.groupby("city", as_index=False).agg(matches_played=("match_id", "count"), avg_runs=("total_runs", "mean")).sort_values("avg_runs", ascending=False)

season_trends = unified_df.groupby("season", as_index=False)["total_runs"].sum().sort_values("season")

calc_winner = team_scores.sort_values(["match_id", "total_runs"], ascending=[True, False]).drop_duplicates("match_id").rename(columns={"batting_team": "calculated_winner"})
win_validation = calc_winner.merge(matches_df[["id", "winner"]], left_on="match_id", right_on="id", how="left")
win_validation["is_accurate"] = win_validation["calculated_winner"].astype(str) == win_validation["winner"].astype(str)

print("Stage 4 core analysis completed successfully.")

Stage 4 core analysis completed successfully.


### Task 5: Derived Insights

In [31]:
def first_value(frame, column, default="N/A"):
    if frame.empty:
        return default
    return frame.iloc[0][column]

most_consistent_batter = first_value(consistent_batters, "batter")
best_death_over_team = first_value(death_overs_teams, "batting_team")
best_venue = first_value(venue_avg, "venue")
best_city = first_value(city_avg, "city")
top_batter_name = first_value(top_batters, "batter")
top_batter_runs = first_value(top_batters, "batsman_runs", 0)

insights = f"""Insights:

Toss: {toss_win_pct:.0f}%
MVP: Batter ({pom_top_batter_count}) | Bowler ({pom_top_bowler_count})
Death: {best_death_over_team}
Venue: {best_venue} ({best_city})
Consistent: {most_consistent_batter}
Scorer: {top_batter_name} ({top_batter_runs})
"""

print(insights)

Insights:

Toss: 51%
MVP: Batter (496) | Bowler (242)
Death: Mumbai Indians
Venue: Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam (Bengaluru)
Consistent: KL Rahul
Scorer: V Kohli (8014)



### Task 6: Reporting

In [32]:
def clean_columns(frame):
    frame.columns = [str(column).strip().lower().replace(" ", "_") for column in frame.columns]
    return frame

for frame_name in ["runs_per_match", "team_scores", "top_batters", "strike_rate_df", "economy_df", "consistent_batters", "highest_score", "top_boundary_hitters", "boundary_pct_df", "top_dot_bowlers", "runs_per_over", "powerplay_teams", "death_overs_teams", "death_overs_batters", "inning_distribution", "toss_runs_comparison", "venue_avg", "city_avg", "season_trends", "win_validation"]:
    globals()[frame_name] = clean_columns(globals()[frame_name])

top_batters = top_batters.sort_values("batsman_runs", ascending=False)
strike_rate_df = strike_rate_df.sort_values("strike_rate", ascending=False)
economy_df = economy_df.sort_values("economy_rate")
consistent_batters = consistent_batters.sort_values("avg_runs", ascending=False)
death_overs_teams = death_overs_teams.sort_values("total_runs", ascending=False)
death_overs_batters = death_overs_batters.sort_values("batsman_runs", ascending=False)
venue_avg = venue_avg.sort_values("avg_runs", ascending=False)
city_avg = city_avg.sort_values("avg_runs", ascending=False)
season_trends = season_trends.sort_values("season")

print("DataFrames formatted for reporting.")

DataFrames formatted for reporting.


### Task 7: Data Export

In [33]:
from pathlib import Path
import csv

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)


def export_csv(frame, filename):
    path = output_dir / filename
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(frame.columns.tolist())
        writer.writerows(frame.itertuples(index=False, name=None))


export_csv(runs_per_match, "runs_per_match.csv")
export_csv(top_batters, "top_batters.csv")
export_csv(strike_rate_df, "strike_rate.csv")
export_csv(economy_df, "economy.csv")
export_csv(team_scores, "team_scores.csv")
export_csv(death_overs_teams, "death_overs.csv")

print("Export completed.")

Export completed.
